# Baseline SLM Evaluation: Qwen2.5-1.5B-Instruct
**Project:** Python Learning Assistant Using Small Language Model (SLM)  
**Notebook:** `notebooks/baseline_evaluation.ipynb`  
**Target Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Unadapted Pretrained Baseline)  
**Evaluation Dataset:** `data/processed/test.jsonl` (38 Held-Out Test Questions)  

This notebook evaluates the unadapted baseline model across the held-out test split, capturing:
1. Exact model responses.
2. Inference latency per question (seconds).
3. Generation throughput (tokens/second).
4. Memory consumption (VRAM / RAM).
5. AST code syntax validity on generated code snippets.

In [ ]:
# Step 1: Hardware Pre-Flight & GPU Auditing
import torch
import psutil
import os

print("=== Runtime Hardware Audit ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA GPU Detected : {gpu_name}")
    print(f"Available VRAM    : {vram_gb:.2f} GB")
    device = "cuda"
    major, _ = torch.cuda.get_device_capability(0)
    dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
else:
    cpu_count = os.cpu_count()
    ram_gb = psutil.virtual_memory().total / (1024**3)
    avail_gb = psutil.virtual_memory().available / (1024**3)
    print(f"No CUDA GPU detected. Running on CPU ({cpu_count} logical cores).")
    print(f"System RAM: {ram_gb:.2f} GB Total, {avail_gb:.2f} GB Available.")
    device = "cpu"
    dtype = torch.float32

print(f"Active Device: {device}, Precision: {dtype}")

In [ ]:
# Step 2: Install Hugging Face Ecosystem Dependencies
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 psutil pyyaml

In [ ]:
# Step 3: Load Model and Tokenizer (Singleton Initialization)
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading tokenizer and model for '{MODEL_ID}'...")

load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)
if device == "cuda":
    model = model.to("cuda")
model.eval()

load_time = time.perf_counter() - load_start
print(f"Model successfully loaded in {load_time:.2f} seconds.")

In [ ]:
# Step 4: Define System Prompt and Inference Pipeline
SYSTEM_PROMPT = """You are an expert, patient, and friendly Python Learning Assistant.
Your mission is to help beginner students master Python programming.
Follow these principles:
1. Explain concepts in clear, simple English without unnecessary jargon.
2. Provide idiomatic, clean, and bug-free Python code examples.
3. Break down code step-by-step or line-by-line when explaining snippets.
4. When diagnosing errors, clearly explain the cause and provide the exact fix.
5. Always encourage the student with beginner-friendly practice questions and hints.
"""

def generate_slm_response(instruction, input_text="", max_new_tokens=512, temperature=0.7, top_p=0.9, repetition_penalty=1.1):
    user_content = instruction.strip()
    if input_text and input_text.strip():
        user_content += f"\n\n```python\n{input_text.strip()}\n```"

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT.strip()}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    if device == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    input_tokens = inputs["input_ids"].shape[1]

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=(temperature > 0.0),
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    latency = time.perf_counter() - t0

    gen_tokens = outputs[0][input_tokens:]
    out_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    tok_per_sec = len(gen_tokens) / max(latency, 0.001)

    return {
        "response": out_text,
        "latency_seconds": round(latency, 3),
        "input_tokens": input_tokens,
        "output_tokens": len(gen_tokens),
        "tokens_per_second": round(tok_per_sec, 2)
    }

In [ ]:
# Step 5: Execute Evaluation Across Held-Out Test Set (38 cases)
import json
from pathlib import Path

test_file = Path("data/processed/test.jsonl")
if not test_file.exists():
    test_file = Path("data/test.jsonl")

test_cases = []
with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_cases.append(json.loads(line))

print(f"Loaded {len(test_cases)} held-out test cases.")

results = []
for idx, tc in enumerate(test_cases, 1):
    inst = tc["instruction"]
    inp = tc.get("input", "")
    gt = tc.get("output", "")
    
    print(f"[{idx}/{len(test_cases)}] Evaluating: {inst[:45]}...")
    out = generate_slm_response(inst, inp)
    
    results.append({
        "test_case_id": idx,
        "instruction": inst,
        "input": inp,
        "ground_truth": gt,
        "model_response": out["response"],
        "latency_seconds": out["latency_seconds"],
        "input_tokens": out["input_tokens"],
        "output_tokens": out["output_tokens"],
        "tokens_per_second": out["tokens_per_second"],
        "status": "success"
    })

print("Evaluation complete!")

In [ ]:
# Step 6: Summary Metrics and Export to experiments/baseline_results.json
latencies = [r["latency_seconds"] for r in results]
tokens = [r["output_tokens"] for r in results]
tps = [r["tokens_per_second"] for r in results]

summary = {
    "evaluation_name": "Baseline SLM Evaluation (Unadapted Qwen2.5-1.5B-Instruct)",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "model_id": MODEL_ID,
    "device": device,
    "precision": str(dtype),
    "total_test_cases": len(results),
    "successful_cases": len(results),
    "failed_cases": 0,
    "average_latency_seconds": round(sum(latencies) / len(latencies), 3),
    "median_latency_seconds": round(sorted(latencies)[len(latencies)//2], 3),
    "average_tokens_generated": round(sum(tokens) / len(tokens), 1),
    "average_tokens_per_second": round(sum(tps) / len(tps), 2),
    "test_results": results
}

out_dir = Path("experiments")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} evaluated test cases to experiments/baseline_results.json")
print(f"Average Latency: {summary['average_latency_seconds']}s | Average Speed: {summary['average_tokens_per_second']} tok/s")